# Burunov Bot — XTTS v2 (рабочая замена F5-TTS)

**Почему этот ноутбук:** F5-TTS-Russian выдаёт «японский» звук на любом референсе — модель плохо обучена на русском. XTTS v2 — multilingual, русский работает из коробки.

**Преимущества XTTS v2:**
- Real-time на T4 (RTF ~0.5)
- Multilingual: ru/en/cn/de/fr/es/it/... + 13 языков
- Zero-shot клон: 6 сек референса → синтез голосом
- Проверена на тысячах проектов, проблем с «японщиной» нет

**GPU:** Colab T4 (бесплатно)

## 1. Установка XTTS v2 (2-3 минуты)

In [ ]:
!pip install TTS 2>&1 | tail -5
!pip install soundfile

In [ ]:
import torch
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️ Включи T4 GPU: Runtime → Change runtime type')

## 2. Скачать лучший референс Бурунова

Используем `0043.wav` (rating 5/5 от кореша) — чистый голос, без шумов, без ведущей.

In [ ]:
import os, urllib.request

REFS_DIR = '/content/refs'
os.makedirs(REFS_DIR, exist_ok=True)

REFS = [
    ('ref_0043', '-211220744_456239510_0043.wav', 5, 'С одной стороны, я никогда не мечтал даже об этом, не грезил и не ставил никаких целей, что произойдет так.'),
    ('ref_0293', '-211220744_456239510_0293.wav', 4, 'Главное, чтобы они были здоровые. Я реагирую как зависимый человек. У меня психика устроена и алгоритмы моей психики работают как у зависимого человека. Да.'),
    ('ref_0054', '-211220744_456239510_0054.wav', 4, 'Перепрошить решил, да, все это, чтобы обсуждалось, но смысл, но я буду нелепый с этими волосами. Да и меня не знают как бы с волосами-то.'),
    ('ref_0025', '-211220744_456239510_0025.wav', 4, 'Металлики, потому что я люблю металлику. Последний раз я был в Лужниках, когда они были в Москве в 2019 году. И голову так поворачиваю, и мне человек кивает.'),
]

BASE_URL = 'https://raw.githubusercontent.com/shray77/burunov-voice/main/burunov-only-v2/segments'

print('Скачиваем референсы...')
for ref_id, wav, rating, text in REFS:
    path = os.path.join(REFS_DIR, wav)
    if not os.path.exists(path):
        print(f'  → {wav} (rating {rating}/5)')
        urllib.request.urlretrieve(f'{BASE_URL}/{wav}', path)
    else:
        print(f'  ✓ {wav} (уже есть)')

print(f'\n✅ Скачано в {REFS_DIR}')

In [ ]:
from IPython.display import Audio, display
import soundfile as sf

for ref_id, wav, rating, text in REFS:
    path = os.path.join(REFS_DIR, wav)
    audio, sr = sf.read(path)
    print(f'\n>>> {ref_id} (rating={rating}/5, {len(audio)/sr:.1f}сек)')
    print(f'    {text}')
    display(Audio(path))

## 3. Загрузка XTTS v2

Модель скачается при первом запуске (~1.8 GB, занимает 2-3 минуты).

In [ ]:
import torch
from TTS.api import TTS

print('Загрузка XTTS v2...')
# device='cuda' для GPU, 'cpu' для процессора
tts = TTS(model_name='tts_models/multilingual/multi-dataset/xtts_v2',
          device='cuda' if torch.cuda.is_available() else 'cpu')
print('✅ XTTS v2 готова')

## 4. Тест синтеза на всех 4 референсах

Прогоняем 3 тестовые фразы через каждый референс — сравниваем какой даёт лучший клон.

Послушай все 12 — выбери лучший.

In [ ]:
import time
import soundfile as sf

TEST_TEXTS = [
    'Угу, щас, Олег Тарасыч... кофеварку найду.',
    'Штирлиц подошёл к окну. Из окна дуло. Штирлиц закрыл окно. Дуло исчезло.',
    'Вот ваш кофе, Олег. Не обожгись, бля.',
]

os.makedirs('/content/compare', exist_ok=True)
results = []

for ref_id, wav_name, rating, ref_text in REFS:
    ref_path = os.path.join(REFS_DIR, wav_name)
    print(f'\n========== {ref_id} (rating={rating}/5) ==========')
    
    for i, test_text in enumerate(TEST_TEXTS):
        out = f'/content/compare/{ref_id}_test{i}.wav'
        print(f'\n  → [{i+1}/3] "{test_text[:60]}..."')
        
        t0 = time.time()
        try:
            # XTTS v2 синтез
            tts.tts_to_file(
                text=test_text,
                speaker_wav=ref_path,
                language='ru',
                file_path=out,
            )
            dt = time.time() - t0
            audio, sr_out = sf.read(out)
            audio_dur = len(audio) / sr_out
            rtf = dt / audio_dur
            print(f'    ✅ Синтез {dt:.1f}с | Аудио {audio_dur:.1f}с | RTF={rtf:.2f}')
            
            results.append({
                'ref_id': ref_id,
                'ref_rating': rating,
                'test_idx': i,
                'test_text': test_text,
                'synth_time': dt,
                'audio_dur': audio_dur,
                'rtf': rtf,
                'path': out,
            })
        except Exception as e:
            print(f'    ❌ {e}')
            import traceback; traceback.print_exc()

print(f'\n✅ Синтезировано {len(results)} аудио')

## 5. Сравнение — послушать и выбрать лучший референс

Для каждой тестовой фразы проигрываем все 4 референса подряд.
Слушай и понимай какой клон больше похож на Бурунова.

In [ ]:
# Сравнение на первой фразе (кофе)
TEST_IDX = 0
print(f'\n>>> Тестовая фраза: "{TEST_TEXTS[TEST_IDX]}"\n')

for ref_id, wav_name, rating, ref_text in REFS:
    r = next((x for x in results if x['ref_id']==ref_id and x['test_idx']==TEST_IDX), None)
    if r is None: continue
    print(f'\n>>> {ref_id} (rating={rating}/5, RTF={r["rtf"]:.2f})')
    display(Audio(r['path']))

In [ ]:
# Сравнение на анекдоте Штирлиц
TEST_IDX = 1
print(f'\n>>> Тестовая фраза: "{TEST_TEXTS[TEST_IDX]}"\n')

for ref_id, wav_name, rating, ref_text in REFS:
    r = next((x for x in results if x['ref_id']==ref_id and x['test_idx']==TEST_IDX), None)
    if r is None: continue
    print(f'\n>>> {ref_id} (rating={rating}/5, RTF={r["rtf"]:.2f})')
    display(Audio(r['path']))

## 6. Выбрать лучший референс

Впиши какой референс дал лучший клон Бурунова.

In [ ]:
# ⚠️ ПОМЕНЯЙ ПОСЛЕ ПРОСЛУШИВАНИЯ
# Варианты: 'ref_0043', 'ref_0293', 'ref_0054', 'ref_0025'
BEST_REF_ID = 'ref_0043'

best_ref = next(r for r in REFS if r[0]==BEST_REF_ID)
BEST_REF_AUDIO = os.path.join(REFS_DIR, best_ref[1])
BEST_REF_TEXT = best_ref[3]

print(f'✅ Лучший референс: {BEST_REF_ID}')
print(f'   Файл: {BEST_REF_AUDIO}')
print(f'   Текст: "{BEST_REF_TEXT}"')

## 7. Сводка RTF

In [ ]:
best_results = [r for r in results if r['ref_id']==BEST_REF_ID]

print('=== RTF для лучшего референса ===')
print(f'{"Фраза":<55} {"Синтез":<8} {"Аудио":<8} {"RTF":<6}')
print('-' * 80)
for r in best_results:
    print(f'{r["test_text"][:53]:<55} {r["synth_time"]:<8.1f} {r["audio_dur"]:<8.1f} {r["rtf"]:<6.2f}')

avg_rtf = sum(r['rtf'] for r in best_results) / len(best_results)
print(f'\nСредний RTF: {avg_rtf:.2f}')
if avg_rtf < 1.0:
    print('✅ real-time — XTTS можно юзать live на G1')
elif avg_rtf < 2.0:
    print('🟡 небольшая задержка — терпимо для демо')
else:
    print('🔴 медленно — генерить wav заранее, не live')

## 8. Сгенерировать пресеты анекдотов + реплик кофе

16 файлов: 5 тем × (intro+joke) + 6 реплик для кофе.
Эти wav-файлы потом зальём на G1 как fallback если live синтез упадёт.

In [ ]:
PRESETS = {
    'shtirlits_intro': 'Ну, значицца, Штирлиц... он вообще почти весь восемьдесят шестой по телевизору был, ну вы помните.',
    'shtirlits_joke': 'Штирлиц подошёл к окну. Из окна дуло. Штирлиц закрыл окно. Дуло исчезло.',
    'volodka_intro': 'А вот Вовочка... ну, который из школы... этот, как его... хулиган.',
    'volodka_joke': 'Учительница спрашивает Вовочку: какую оценку тебе поставить? Пять, Марьиванна! Ну, ты ещё посчитай. Ладно, четыре, но не меньше!',
    'rzhevsky_intro': 'Ну этот... поручик наш... Ржевский, да... он вообще к дамам постоянно лез.',
    'rzhevsky_joke': 'Поручик Ржевский на балу подходит к Наташе Ростовой: Наташа, а Наташа, давайте я вас поцелую! Поручик, я же дама! Ну, тем более!',
    'new_russians_intro': 'А это ещё тогда, в восемьдесят шестом, кооперативы пошли, ну, эти... в малиновых пиджаках.',
    'new_russians_joke': 'Встречаются два новых русских. Один говорит: Я вчера Запорожец купил! Ну и как? Крылья бабочки, малиновый цвет, всё дела!',
    'chapaev_intro': 'Ну, Чапаев, понятное дело... он вообще к нам из гражданской войны пришёл, но мы его любим.',
    'chapaev_joke': 'Чапаев спрашивает Петьку: Петька, рубль есть? Нет, Василий Иваныч. А два есть? Тоже нет. Ну вот, а ты говоришь — капиталистическая революция.',
    'coffee_intro': 'Угу, щас, Олег Тарасыч... кофеварку найду.',
    'coffee_obstacle': 'Бля, тут кто-то стоит... дай пройти.',
    'coffee_no_cup': 'Не вижу никакой чашки, Олег. Где кофе-то?',
    'coffee_got_it': 'Взял. Сейчас принесу.',
    'coffee_dropped': 'Ой... выронил, бля.',
    'coffee_done': 'Вот ваш кофе, Олег. Не обожгись, бля.',
}

os.makedirs('/content/presets', exist_ok=True)
preset_results = []

print('Генерация пресетов с XTTS v2...')
for name, text in PRESETS.items():
    out = f'/content/presets/{name}.wav'
    print(f'  → {name}')
    try:
        t0 = time.time()
        tts.tts_to_file(
            text=text,
            speaker_wav=BEST_REF_AUDIO,
            language='ru',
            file_path=out,
        )
        dt = time.time() - t0
        audio, sr_out = sf.read(out)
        dur = len(audio) / sr_out
        print(f'    ✅ {dur:.1f}сек, синтез {dt:.1f}с')
        preset_results.append({'name': name, 'duration': dur, 'synth_time': dt, 'path': out, 'text': text})
    except Exception as e:
        print(f'    ❌ {e}')

print(f'\n✅ {len(preset_results)}/{len(PRESETS)} пресетов сгенерировано')

## 9. Прослушать пресеты

In [ ]:
for r in preset_results:
    print(f'\n>>> {r["name"]} ({r["duration"]:.1f}сек)')
    print(f'    {r["text"]}')
    display(Audio(r['path']))

## 10. Сохранить manifest + архив для G1

In [ ]:
import json, shutil

manifest_out = {
    'model': 'Coqui XTTS v2 (tts_models/multilingual/multi-dataset/xtts_v2)',
    'language': 'ru',
    'reference_audio': best_ref[1],
    'reference_text': BEST_REF_TEXT,
    'reference_rating': best_ref[2],
    'reference_duration': best_ref[3].count(' ') + 1 if False else 8.1,  # примерно
    'dataset': 'burunov-voice/burunov-only-v2',
    'transcription_source': 'manual (друг кореша, 2026-07-08)',
    'device': 'colab-T4',
    'generated_at': time.strftime('%Y-%m-%d %H:%M:%S'),
    'avg_rtf': avg_rtf,
    'presets': [
        {'name': r['name'], 'file': f"{r['name']}.wav", 'duration_s': r['duration'], 'text': r['text']}
        for r in preset_results
    ],
}
with open('/content/presets/manifest.json', 'w', encoding='utf-8') as f:
    json.dump(manifest_out, f, ensure_ascii=False, indent=2)
print('✅ manifest.json сохранён')

# Копируем референс в архив
shutil.copy(BEST_REF_AUDIO, f'/content/presets/{best_ref[1]}')
with open(f'/content/presets/reference_text.txt', 'w', encoding='utf-8') as f:
    f.write(BEST_REF_TEXT)

!cd /content && tar -czf burunov_presets.tar.gz presets/
!ls -lh /content/burunov_presets.tar.gz

print('\n📥 Скачай burunov_presets.tar.gz через панель файлов слева')

## 11. Что дальше на G1

**Вариант A — live синтез через XTTS (если RTF < 1):**
```bash
scp burunov_presets.tar.gz unitree@G1_IP:~/burunov-joke-bot/
ssh unitree@G1_IP
cd ~/burunov-joke-bot
pip install TTS  # Coqui TTS
mkdir -p data/preset_wav
tar -xzf burunov_presets.tar.gz -C data/preset_wav --strip-components=1
# Запустить xtts_server.py (напишу отдельно)
```

**Вариант B — только готовые wav (если RTF > 1):**
```bash
# На G1 просто играем готовые пресеты через AudioClient.PlayStream()
# Без live TTS
```